# Prompt Engineering Techniques

## Few-Shot Prompting

Provide $k$ input-output examples before the actual query. The model learns the pattern in-context:

$$P(y \mid x, e_1, ..., e_k) > P(y \mid x)$$

```
Translate English to French:
sea otter => loutre de mer
cheese    => fromage
coffee    => [MODEL COMPLETES]
```

**Tips**: Order matters (recency bias), use diverse examples, keep format consistent.

---

## Chain-of-Thought (CoT)

Encouraging the model to reason step-by-step before answering significantly improves performance on multi-step tasks.

### Standard CoT (Few-shot)
```
Q: Roger has 5 balls. He buys 2 cans of 3 balls each. How many does he have?
A: Roger started with 5. 2 cans × 3 = 6 new balls. 5 + 6 = 11. Answer: 11

Q: [New question]
A: [Model reasons step-by-step]
```

### Zero-Shot CoT
Simply append: **"Let's think step by step."**

This single phrase was shown to unlock reasoning without any examples (Kojima et al., 2022).

---

## Self-Consistency

Sample $m$ reasoning paths, then take the majority vote:

$$\hat{y} = \arg\max_y \sum_{i=1}^{m} \mathbf{1}[y_i = y]$$

More reliable than greedy decoding on arithmetic and commonsense tasks.

---

## Tree-of-Thoughts (ToT)

Extends CoT by exploring **multiple reasoning branches** and using search (BFS/DFS) with a value function:

```
Problem
├── Thought A → Evaluate (sure/maybe/impossible)
│   ├── Thought A1
│   └── Thought A2 ← expand
└── Thought B → Evaluate
    └── ...
```

Uses a **state evaluator** (another LLM call) to prune unpromising branches.

---

## ReAct (Reason + Act)

Interleaves reasoning traces with actions (tool calls):

$$\text{Thought}_t \to \text{Action}_t \to \text{Observation}_t \to \text{Thought}_{t+1} \to ...$$

```
Thought: I need to find the population of Paris.
Action: Search["Paris population 2024"]
Observation: Paris has ~2.1 million people in the city proper.
Thought: Now I can answer the question.
Answer: Paris has approximately 2.1 million inhabitants.
```

---

## Other Techniques

| Technique | Core Idea |
|-----------|----------|
| **Program-of-Thought (PoT)** | Generate code instead of text reasoning; execute it |
| **Least-to-Most** | Decompose → solve subproblems sequentially |
| **Step-Back** | First ask a general principle, then apply it |
| **Generated Knowledge** | Generate relevant facts first, then answer |
| **Analogical Reasoning** | Ask model to recall similar problems, then solve |
| **Directional Stimulus** | Provide a hint/keyword that steers the answer |
| **Auto-CoT** | Auto-generate CoT examples using clustering |

In [1]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def ask(prompt, temperature=0.0, max_tokens=300):
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature, max_tokens=max_tokens
    )
    return r.choices[0].message.content

# ── Zero-Shot CoT ─────────────────────────────────────────────────────────────
problem = "A store sells apples at $0.50 each and oranges at $0.75 each. If I buy 6 apples and 4 oranges, how much do I spend?"

no_cot   = ask(problem)
with_cot = ask(problem + "\n\nLet's think step by step.")

print("Without CoT:", no_cot)
print("\nWith CoT:", with_cot)

In [2]:
# ── Self-Consistency ──────────────────────────────────────────────────────────
from collections import Counter

def self_consistency(problem, m=5):
    answers = []
    for _ in range(m):
        response = ask(
            problem + "\n\nLet's think step by step. End with 'Answer: <number>'",
            temperature=0.7
        )
        # extract last answer
        for line in reversed(response.split('\n')):
            if 'Answer:' in line:
                answers.append(line.split('Answer:')[-1].strip())
                break
    
    vote = Counter(answers).most_common(1)[0]
    print(f"Answers: {answers}")
    print(f"Majority vote: {vote[0]} ({vote[1]}/{m} agree)")

self_consistency("What is 17 × 23?")

In [3]:
# ── Few-Shot Prompting ────────────────────────────────────────────────────────
few_shot_prompt = """
Classify the following as a Question, Statement, or Command.

Text: "What time is it?"
Type: Question

Text: "The sky is blue."
Type: Statement

Text: "Close the door."
Type: Command

Text: "Did you finish your homework?"
Type:"""

print(ask(few_shot_prompt))

In [4]:
# ── Program-of-Thought (PoT) ──────────────────────────────────────────────────
pot_prompt = """
Solve this math problem by writing Python code, then execute it mentally.

Problem: A train travels at 120 km/h. How long does it take to travel 450 km?

Write Python code to solve it:
"""

code_response = ask(pot_prompt)
print(code_response)

# Execute the generated code
exec(code_response.split('```python')[-1].split('```')[0] if '```python' in code_response else "")

In [5]:
# ── Step-Back Prompting ───────────────────────────────────────────────────────
specific_q = "What happens to the pressure of a gas when its temperature rises at constant volume?"

step_back_prompt = f"""
Step 1: What is the general physics principle relevant to this question?
Question: {specific_q}
"""

principle = ask(step_back_prompt)
print("Principle:", principle)

final = ask(f"Principle: {principle}\n\nUsing this principle, answer: {specific_q}")
print("\nAnswer:", final)

## Additional Learning Resources

### Papers
- [Chain-of-Thought Prompting (Wei et al., 2022)](https://arxiv.org/abs/2201.11903)
- [Zero-Shot CoT (Kojima et al., 2022)](https://arxiv.org/abs/2205.11916)
- [Self-Consistency (Wang et al., 2022)](https://arxiv.org/abs/2203.11171)
- [Tree of Thoughts (Yao et al., 2023)](https://arxiv.org/abs/2305.10601)
- [ReAct (Yao et al., 2022)](https://arxiv.org/abs/2210.03629)
- [Program of Thoughts (Chen et al., 2022)](https://arxiv.org/abs/2211.12588)
- [Least-to-Most Prompting (Zhou et al., 2022)](https://arxiv.org/abs/2205.10625)
- [Step-Back Prompting (Zheng et al., 2023)](https://arxiv.org/abs/2310.06117)

### Guides
- [Prompt Engineering Guide Techniques](https://www.promptingguide.ai/techniques)
- [Learn Prompting Advanced](https://learnprompting.org/docs/advanced)